Qt Property 的“三件套”是构建一个**可被 Qt 元对象系统（MOC）识别、支持数据绑定和 QML 交互**的响应式属性的最小完备单元。缺少任何一件，属性就会退化为普通变量或失去核心能力。

以下是三件套的逐一拆解：

### 1. Storage（内部存储）
> **是什么**：一个以 `_` 开头的私有实例变量，用于真正保存数据。
> **示例**：`self._name = ""`

-   **职责**：作为数据的唯一真实来源（Single Source of Truth）。Getter 从这里读，Setter 往这里写。
-   **为什么必须是私有的**：防止外部代码绕过 Setter 直接修改值。如果外部直接执行 `obj._name = "xxx"`，就不会触发等值守卫，也不会发射 Notify 信号，导致 UI 与模型状态不一致。
-   **命名约定**：通常使用 `_propertyName` 格式，避免与 Property 本身的公开名称冲突。

### 2. Signal（通知信号）
> **是什么**：一个无参数的 `Signal()`，在属性值发生有效变化时发射。
> **示例**：`nameChanged = Signal()`

-   **职责**：充当属性变化的**广播器**。它是连接 Model 和 View（或任何其他监听者）的唯一纽带。
-   **命名约定**：**必须**是 `<propertyName>Changed` 格式（如 `name` → `nameChanged`）。这是 Qt 的硬性约定，QML 引擎、`QDataWidgetMapper`、自定义 Binder 都依赖这个命名规则自动查找并连接信号。
-   **关键约束**：
    -   只在值**真正改变**后发射（由 Setter 中的等值守卫保证）。
    -   不携带新值作为参数（监听者收到信号后自行调用 Getter 获取最新值），这避免了信号签名与属性类型耦合的问题。

### 3. Property（属性声明）
> **是什么**：通过 `@Property(type, notify=signal)` 装饰器/宏注册的元数据对象。
> **示例**：`@Property(str, notify=nameChanged)`

-   **职责**：将 Storage 和 Signal **粘合**在一起，并向 Qt 元对象系统注册。它做了三件关键事：
    1.  **声明类型**：告诉 MOC/QML 该属性的数据类型（`str`, `int`, `float` 等），实现跨语言类型安全。
    2.  **关联读写器**：指定哪个函数是 Getter、哪个是 Setter，使 `obj.property("name")` 和 `obj.setProperty("name", value)` 能正确路由。
    3.  **绑定通知信号**：通过 `notify=` 参数将 Signal 与属性永久关联，使框架知道"当这个信号发射时，代表这个属性变了"。

### 三件套的协作流程

```text
外部赋值 obj.name = "张三"
        │
        ▼
   ┌─────────┐    值相同？    ┌──────────┐
   │  Setter  │ ──────────► │ 直接返回  │ (不写存储、不发信号)
   └─────────┘               └──────────┘
        │ 值不同
        ▼
   ┌─────────┐
   │ Storage │  self._name = "张三"
   └─────────┘
        │
        ▼
   ┌─────────┐
   │  Signal │  self.nameChanged.emit()
   └─────────┘
        │
        ▼
   ┌─────────────────────────────┐
   │ 所有连接到 nameChanged 的槽 │
   │ • QML Text { text: model.name }
   │ • Binder → widget.setText()
   │ • PropertyAnimation
   └─────────────────────────────┘
```

### ⚠️ 缺少任何一件的后果

| 缺失项 | 后果 |
| :--- | :--- |
| 缺 Storage | Getter/Setter 无处读写，或被迫用全局变量/字典，破坏封装 |
| 缺 Signal | 属性变成"哑巴"，UI 无法自动刷新，QML 绑定失效，等同于普通 Python 属性 |
| 缺 Property | 数据仍在、信号仍发，但 **Qt 元对象系统完全不知道它的存在**：QML 不可见、`property()` 反射失败、动画无法驱动、Designer 不显示 |

### 💡 为什么之前要用 `qt_property` 描述符？

正是因为三件套是**机械性、重复性、且强约定**的样板代码。每增加一个属性就要手写 ~13 行结构完全相同的代码，极易出错（忘发信号、拼错信号名、漏掉等值守卫）。`qt_property` 描述符的本质就是**在类创建阶段自动生成这三件套**，将 13 行压缩为 1 行，同时从机制上杜绝遗漏。